In [2]:
import mlflow
import pandas as pd 
import numpy as np
import mlflow.sklearn

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [30]:
df = pd.read_csv('IMDB.csv')
df.head()

,review,sentiment
0,Film version of Sandra Bernhard's one-woman of...,negative
1,I switched this on (from cable) on a whim and ...,positive
2,The `plot' of this film contains a few holes y...,negative
3,"Some amusing humor, some that falls flat, some...",negative
4,What can you say about this movie? It was not ...,negative


In [31]:
data = df.sample(500)
data.to_csv('data.csv', index=False)

In [32]:
data.head()

,review,sentiment
951,"I am usually a big fan of Pacino (Scarface, Se...",negative
949,I was particularly moved by the understated co...,positive
811,I've always been enthusiastic about period dra...,negative
102,This film is a brilliant retelling of Shakespe...,positive
716,This movie was an attempt to go into places mo...,negative


# Data Preprocessing

In [11]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Anvi\AppData\Roaming\nltk_data...


True

In [33]:
# text preprocessing functions

def lemmatization(text):
    """ Lemmatize the input text using WordNetLemmatizer from NLTK.
    Args:        text (str): The input text to be lemmatized.
    Returns:     str: The lemmatized text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """ Remove stop words from the input text.
    Args:        text (str): The input text from which to remove stop words.
    Returns:     str: The text with stop words removed."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """ Remove numbers from the input text.
    Args:        text (str): The input text from which to remove numbers.
    Returns:     str: The text with numbers removed."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """ Convert the input text to lowercase.
    Args:        text (str): The input text to be converted to lowercase.
    Returns:     str: The text converted to lowercase."""
    text = text.split()
    text = [word.lower() for word in text]      
    return ' '.join(text)

def removing_punctuations(text):
    """ Remove punctuation from the input text.
    Args:        text (str): The input text from which to remove punctuation.
    Returns:     str: The text with punctuation removed."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """ Remove URLs from the input text.
    Args:        text (str): The input text from which to remove URLs.
    Returns:     str: The text with URLs removed."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)   

def normalize_text(df):
    """ Normalize the input text by applying a series of preprocessing steps.
    Args:        text (str): The input text to be normalized.
    Returns:     str: The normalized text after applying all preprocessing steps."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error in normalize_text: {e}")
        raise

In [34]:
data = normalize_text(data)
data.head()

,review,sentiment
951,usually big fan pacino scarface serpico devil ...,negative
949,particularly moved understated courage integri...,positive
811,always enthusiastic period drama art form bbc ...,negative
102,film brilliant retelling shakespeare s classic...,positive
716,movie attempt go place perhaps venture into si...,negative


In [35]:
data['sentiment'].value_counts()

sentiment
negative    259
positive    241
Name: count, dtype: int64

In [36]:
x = data['sentiment'].isin(['positive', 'negative'])
data = data[x]  
data['sentiment'].value_counts()

sentiment
negative    259
positive    241
Name: count, dtype: int64

In [37]:
data['sentiment'] = data['sentiment'].map({'positive': 1, 'negative': 0})
data.head()

,review,sentiment
951,usually big fan pacino scarface serpico devil ...,0
949,particularly moved understated courage integri...,1
811,always enthusiastic period drama art form bbc ...,0
102,film brilliant retelling shakespeare s classic...,1
716,movie attempt go place perhaps venture into si...,0


In [38]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [48]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(data['review'])
y = data['sentiment']


In [49]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25 random_state=42)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (3062658910.py, line 1)

In [50]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/candobettercode/mlops-aws-nlp-project.mlflow')
dagshub.init(repo_owner='candobettercode', repo_name='mlops-aws-nlp-project', mlflow=True)

mlflow.set_experiment("Sentiment Analysis Baseline Experiment")

2026-05-07 23:47:05,311 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/candobettercode/mlops-aws-nlp-project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "candobettercode/mlops-aws-nlp-project"

2026-05-07 23:47:05,328 - INFO - Initialized MLflow to track repo "candobettercode/mlops-aws-nlp-project"


Repository candobettercode/mlops-aws-nlp-project initialized!

2026-05-07 23:47:05,328 - INFO - Repository candobettercode/mlops-aws-nlp-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/7a17f983ddde495584911bdafc24749d', creation_time=1778176078204, experiment_id='0', last_update_time=1778176078204, lifecycle_stage='active', name='Sentiment Analysis Baseline Experiment', tags={}, trace_location=None, workspace='default'>

In [52]:
import mlflow
import logging
import os
import time

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

# configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("Starting Mlflow run...")

with mlflow.start_run(run_name="Logistic Regression Baseline") as run:
    start_time = time.time()

    try:
        logging.info("Data preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("max_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Data preprocessing completed in %.2f seconds", time.time() - start_time)

        # Train the model
        logging.info("Training Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)
        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model fitting completed")
        
        logging.info("logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")
        
        # Make predictions
        logging.info("Making predictions on the test set...")
        y_pred = model.predict(X_test)

        # Calculate metrics
        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Log metrics to Mlflow
        logging.info("Logging metrics to Mlflow...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-07 23:47:27,422 - INFO - Starting Mlflow run...
2026-05-07 23:47:28,133 - INFO - Data preprocessing parameters...
2026-05-07 23:47:29,068 - INFO - Data preprocessing completed in 0.94 seconds
2026-05-07 23:47:29,068 - INFO - Training Logistic Regression model...
2026-05-07 23:47:29,068 - INFO - Fitting the model...
2026-05-07 23:47:29,096 - INFO - Model fitting completed
2026-05-07 23:47:29,097 - INFO - logging model parameters...
2026-05-07 23:47:29,386 - INFO - Making predictions on the test set...
2026-05-07 23:47:29,386 - INFO - Calculating evaluation metrics...
2026-05-07 23:47:29,386 - INFO - Logging metrics to Mlflow...
2026-05-07 23:47:30,582 - INFO - Saving and logging the model...
2026/05/07 23:47:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 23:47:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object seri

🏃 View run Logistic Regression Baseline at: https://dagshub.com/candobettercode/mlops-aws-nlp-project.mlflow/#/experiments/0/runs/93a4057871074b789e0472e5df76d65d
🧪 View experiment at: https://dagshub.com/candobettercode/mlops-aws-nlp-project.mlflow/#/experiments/0
